# Peptide Designer HF Landscape Bundle

Lean export notebook for creating the structured Hugging Face bundle at `landscapes/dbaasp_amp_v1/`.

This notebook uses a local external DBAASP-style activity CSV to compute aggregate node-level landscape values. Raw peptide rows are not saved to the bundle and are not uploaded.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import json
import os
import sys

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from scripts.export_peptide_landscape_bundle import export_bundle

print(f"Project root: {project_root}")

## Configure Inputs

Set `DBAASP_ACTIVITY_CSV` in the environment or assign `ACTIVITY_CSV` below to a local external CSV path. Keep that file outside Git tracking.

In [ ]:
ACTIVITY_CSV = os.environ.get("DBAASP_ACTIVITY_CSV") or ""

if not ACTIVITY_CSV:
    raise ValueError("Set DBAASP_ACTIVITY_CSV or assign ACTIVITY_CSV to a local external activity CSV path.")

args = SimpleNamespace(
    activity_csv=ACTIVITY_CSV,
    legacy_dir=Path("output/sampling_analysis/peptides_gtm_analysis"),
    bundle_root=Path("output/hf/peptide_designer_data"),
    model_path=Path("models/peptides/model_344000.pt"),
    device=os.environ.get("TORCH_DEVICE", "cpu"),
    decoder_revision=os.environ.get("WAE_PEPTIDES_REVISION", "main"),
    min_observations=1.0,
)

print(f"Legacy artifacts: {project_root / args.legacy_dir}")
print(f"Bundle root: {project_root / args.bundle_root}")
print(f"Device: {args.device}")

## Export Bundle

In [ ]:
bundle_dir = export_bundle(args)
print(bundle_dir)

## Validate Bundle

In [ ]:
import pandas as pd
from safetensors.numpy import load_file

manifest = json.loads((bundle_dir / "landscape.json").read_text())
tensors = load_file(bundle_dir / "landscape.safetensors")
nodes = pd.read_parquet(bundle_dir / "nodes.parquet")

assert manifest["landscape_id"] == "dbaasp_amp_v1"
assert manifest["compatible_decoder_repo"] == "axelrolov/wae_peptides"
assert manifest["raw_source_data_redistributed"] is False
assert "sequence" not in {column.lower() for column in nodes.columns}
assert "source_peptides.parquet" not in [path.name for path in bundle_dir.rglob("*")]

print(f"Tensor keys: {sorted(tensors)}")
print(f"Node rows: {len(nodes):,}")
print(nodes.head())

## Upload to Hugging Face

The upload script reads `HF_TOKEN` or `HUGGING_FACE_HUB_TOKEN`. Run dry-run first, then upload.

In [ ]:
!python scripts/upload_peptide_designer_data.py --dry-run --public

# Uncomment to publish after reviewing the dry-run manifest.
# !python scripts/upload_peptide_designer_data.py --upload --public